# Setup and manual configuration
Mounting Google Drive, cloning the repository from GitHub, setting up input/output paths, and installing specific dependencies inline.
Modify the variables in this cell before starting a new experiment.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive for datasets and results
drive.mount('/content/drive')

# ==========================================
# --- MANUAL EXPERIMENT CONFIGURATION ---
# ==========================================
GIT_REPO_URL = 'https://github.com/EmanuelePietroCometti/SuperSimpleNet.git'
BRANCH_NAME = 'baseline_prove'
REPO_PATH = '/content/SuperSimpleNet'
DATA_PATH = '/content/drive/MyDrive/Tesi/MVTec'
SAVE_PATH = '/content/drive/MyDrive/Tesi/Risultati_SSN/run_custom_01_no_dust'
CATEGORY = 'custom_no_dust'
SETUP_NAME = 'superSimpleNet_baseline'
MODE = 'sup'
# ==========================================

if not os.path.exists(REPO_PATH):
    print(f">>> Cloning branch '{BRANCH_NAME}' from GitHub...")
    # The -b flag forces Git to clone and checkout the specified branch immediately
    !git clone -b {BRANCH_NAME} {GIT_REPO_URL} {REPO_PATH}
else:
    print(f">>> Repository already cloned. Updating branch '{BRANCH_NAME}'...")
    os.chdir(REPO_PATH)
    # Ensure the local repository is explicitly on the target branch and pull updates
    !git checkout {BRANCH_NAME}
    !git pull origin {BRANCH_NAME}

# Change to the working directory
os.chdir(REPO_PATH)
print(f"Working directory set to: {os.getcwd()}")

# Directly create the output folder on Drive
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"Output folder ready on Drive: {SAVE_PATH}")

# Install standard dependencies
!pip install tqdm numpy==1.26.0 anomalib==0.7

# Install specific PyTorch version with CUDA 11.8 support
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 --extra-index-url https://download.pytorch.org/whl/cu118

# Optional: Install wandb for experiment tracking
!pip install wandb

# Insatall onnx
!pip install onnx onnxscript

In [ ]:
import os
import glob
import shutil

# Path configuration consistent with your setup
DATA_PATH = '/content/drive/MyDrive/Tesi/MVTec'
CATEGORY = 'carpet'
NUM_ANOMALIES_PER_DEFECT = 2  # How many anomalous samples to insert into the train set

dataset_root = os.path.join(DATA_PATH, CATEGORY)
test_dir = os.path.join(dataset_root, 'test')
gt_root = os.path.join(dataset_root, 'ground_truth')
train_dir = os.path.join(dataset_root, 'train')

if not os.path.exists(test_dir):
    raise FileNotFoundError(f"Cannot find the test folder in: {test_dir}")

# Identify all defect categories excluding normal images ('good')
defect_types = [d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d)) and d != 'good']

print(f">>> MIXED_SUPERVISION configuration for dataset: {CATEGORY}")
print(f">>> Detected defects: {defect_types}\n")

for defect in defect_types:
    defect_test_dir = os.path.join(test_dir, defect)
    defect_gt_dir = os.path.join(gt_root, defect)
    
    # Destination paths in the training set
    target_train_defect_dir = os.path.join(train_dir, defect)
    target_train_gt_dir = os.path.join(train_dir, 'ground_truth', defect)
    
    os.makedirs(target_train_defect_dir, exist_ok=True)
    os.makedirs(target_train_gt_dir, exist_ok=True)
    
    # Select anomalous images from the test set
    images = sorted(glob.glob(os.path.join(defect_test_dir, "*.png")))
    samples_to_move = images[:NUM_ANOMALIES_PER_DEFECT]
    
    print(f"Processing defect '{defect}': inserting {len(samples_to_move)} samples into the training set...")
    
    for img_path in samples_to_move:
        base_name = os.path.basename(img_path)
        stem = os.path.splitext(base_name)[0]
        
        # Copy the anomalous image into the train folder
        dest_img_path = os.path.join(target_train_defect_dir, base_name)
        if not os.path.exists(dest_img_path):
            shutil.copy(img_path, dest_img_path)
        
        # Look for the corresponding mask in the original ground_truth
        # MVTec uses conventions like '000_mask.png' or '000.png' depending on the release
        mask_candidates = [
            os.path.join(defect_gt_dir, f"{stem}_mask.png"),
            os.path.join(defect_gt_dir, f"{stem}.png")
        ]
        
        mask_copied = False
        for mask_path in mask_candidates:
            if os.path.exists(mask_path):
                dest_mask_path = os.path.join(target_train_gt_dir, os.path.basename(mask_path))
                if not os.path.exists(dest_mask_path):
                    shutil.copy(mask_path, dest_mask_path)
                mask_copied = True
                break
                
        if not mask_copied:
            print(f"⚠️ [WARNING] Mask not found for image {img_path}")

print("\n>>> MVTec dataset successfully prepared locally for SUP mode!")

# Hyperparameter Optimization (Optional)
Launch the fine-tuning script. It dynamically reads the parameters configured in Cell 1.

In [ ]:
import os
os.chdir(REPO_PATH)

print(f">>> Starting Hyperparameter Optimization on dataset: {CATEGORY}...")

!python hyperparameter_finetuning.py \
    --dataset mvtec \
    --category {CATEGORY} \
    --data_path {DATA_PATH} \
    --datasets_folder {DATA_PATH} \
    --results_save_path {SAVE_PATH} \
    --setup_name {SETUP_NAME} \
    --epochs 100 \
    --batch 4

# Training, Evaluation and ONNX Export
This single cell orchestrates the entire pipeline:
1. Runs `train.py` with the specified parameters.
2. Automatically locates the generated `.pt` (or `.ckpt`) weights file on Google Drive.
3. Passes the weights file to `eval.py` and `export_onnx.py`.
4. Organizes the generated ONNX models into a dedicated subfolder.

In [ ]:
import os
import glob
import shutil

os.chdir(REPO_PATH)

print(f">>> Starting Training on {CATEGORY}...")
print(f">>> Saving results directly to: {SAVE_PATH}\n")

# Lancia train.py iniettando le variabili Python
!python train.py \
    --dataset mvtec \
    --category {CATEGORY} \
    --mode {MODE} \
    --data_path {DATA_PATH} \
    --datasets_folder {DATA_PATH} \
    --results_save_path {SAVE_PATH} \
    --setup_name {SETUP_NAME} \
    --num_workers 1 \
    --backbone wide_resnet50_2 \
    --layers layer2 layer3 \
    --image_size 512 512 \
    --epochs 100 \
    --batch 4 \
    --perlin_thr 0.2 \
    --noise_std 0.015 \
    --seg_lr 0.0002 \
    --dec_lr 0.0002 \
    --adapt_lr 0.0001 \
    --patch_size 3 \
    --gamma 0.4 \
    --eval_step_size 5 \
    --seed 42

print(f"\n>>> Searching for the generated weights in {SAVE_PATH}...")
# Cerca ricorsivamente qualsiasi file dei pesi generato dal training
weight_files = glob.glob(os.path.join(SAVE_PATH, "**", "*.pt"), recursive=True) + \
               glob.glob(os.path.join(SAVE_PATH, "**", "*.pth"), recursive=True) + \
               glob.glob(os.path.join(SAVE_PATH, "**", "*.ckpt"), recursive=True)

if not weight_files:
    print("[ERROR] No weights file found. The training process might have failed.")
else:
    # Prende il primo file dei pesi trovato
    WEIGHTS_FILE = weight_files[0]
    print(f">>> Found weights file: {WEIGHTS_FILE}")
    
    print("\n" + "="*40)
    print("--- STARTING EVALUATION ---")
    print("="*40)
    !python eval.py "{WEIGHTS_FILE}" \
    --dataset mvtec \
    --category {CATEGORY} \
    --datasets_folder {DATA_PATH} \
    --results_save_path {SAVE_PATH} \
    --image_size 512 512 \
    --batch 4 \
    --num_workers 1 \
    --seed 42
        
    print("\n" + "="*40)
    print("--- STARTING ONNX EXPORT ---")
    print("="*40)
    !python export_onnx.py "{WEIGHTS_FILE}"
    
    print("\n>>> Reorganizing ONNX files...")
    # Crea una cartella 'onnx' e sposta i modelli esportati
    onnx_dir = os.path.join(SAVE_PATH, "onnx")
    os.makedirs(onnx_dir, exist_ok=True)
    
    for onnx_file in glob.glob(os.path.join(SAVE_PATH, "**", "*.onnx"), recursive=True):
        shutil.move(onnx_file, os.path.join(onnx_dir, os.path.basename(onnx_file)))
        
    print(f"\n>>> Pipeline completed successfully! ONNX models saved in: {onnx_dir}")